# Preparación de Online Retail II
**Lourdes Lema · Progreso 2 · Cuaderno de trabajo v0.1**

Este cuaderno reproduce la integración aprobada y diagnostica la base resultante. Calcula el importe por línea; todavía no define ventas netas o positivas, clientes elegibles ni ventanas del modelo.

**Alcance aprobado:** conservar una copia del bloque idéntico entre hojas. Las repeticiones internas, faltantes, cancelaciones y valores no positivos se mantienen para revisión. No se entrenan modelos.

**Cómo trabajar:** abre este archivo en Google Colab mediante **Archivo → Subir cuaderno**. Ejecuta las celdas de código de arriba hacia abajo con ▶. La celda de carga solicita el Excel original si no lo encuentra. Usa `online_retail_II 2.xlsx`, no el CSV integrado. Puedes guardar una copia del cuaderno en Drive. Esta copia pública no conserva salidas calculadas. Al ejecutar las celdas se producirán los controles a partir del libro cargado.

Fuente: Chen, D. (2012). *Online Retail II* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D . Licencia CC BY 4.0. El original contiene datos de 2009–2011; los resultados no se extrapolan automáticamente al mercado ecuatoriano.

## Entorno y carga del archivo
**Propósito:** disponer de pandas y localizar el archivo original.

**Operaciones:** importar las herramientas; buscar el Excel en la sesión; solicitar la carga si se trabaja en Colab; calcular su huella SHA-256. Esta huella permite comprobar que los bytes del original no cambiaron.

**Resultado esperado:** nombre, tamaño y huella del archivo. La carga no modifica los datos. La huella de referencia identifica el archivo auditado; si difiere, revisaremos la versión antes de reutilizar los conteos.

In [ ]:
from pathlib import Path
import pandas as pd
import openpyxl
import hashlib
import json
import sys
import zipfile

NOMBRE = "online_retail_II 2.xlsx"
candidatos = [Path(NOMBRE), Path("upload") / NOMBRE, Path("/content") / NOMBRE]
ruta = next((p for p in candidatos if p.is_file()), None)
if ruta is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Coloca el Excel original junto al cuaderno o en la carpeta upload.")
    cargados = files.upload()
    opciones = [Path(n) for n in cargados if n.lower().endswith(".xlsx")]
    if len(opciones) != 1:
        raise ValueError("Carga exactamente un archivo XLSX original.")
    ruta = opciones[0]
    del cargados

huella_inicial = hashlib.sha256(ruta.read_bytes()).hexdigest()
HUELLA_REFERENCIA = "30de13d99355445fdaf5ca34e7079794f50b644a0ea05d730faa5afb7f5043b1"
if huella_inicial != HUELLA_REFERENCIA:
    raise ValueError("La huella difiere del Excel auditado. Revisar la versión antes de continuar.")
print("Archivo:", ruta.name)
print("Tamaño en MB:", round(ruta.stat().st_size / 1_000_000, 2))
print("SHA-256:", huella_inicial)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| openpyxl:", openpyxl.__version__)

## Lectura de las dos hojas
**Propósito:** conocer dimensiones y columnas antes de integrar.

**Operaciones:** `ExcelFile` identifica las hojas; `read_excel` lee sus datos; `shape` y `len` cuentan columnas y filas. Los encabezados no se cuentan como registros.

**Resultado esperado:** 525.461 y 541.910 filas, con ocho columnas iguales. La lectura del Excel puede tardar varios minutos. Una fila corresponde a una línea, no necesariamente a una compra independiente.

In [ ]:
libro = pd.ExcelFile(ruta)
assert libro.sheet_names == ["Year 2009-2010", "Year 2010-2011"], "Revisar hojas del archivo."
hojas = {}
for nombre in libro.sheet_names:
    print("Leyendo:", nombre, flush=True)
    hojas[nombre] = pd.read_excel(libro, sheet_name=nombre)
libro.close()
a, b = [hojas[n] for n in libro.sheet_names]
columnas = list(a.columns)
assert columnas == list(b.columns)
assert columnas == ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
resumen_hojas = pd.DataFrame([
    {"Hoja": n, "Filas": len(d), "Columnas": d.shape[1],
     "Primera fecha": d["InvoiceDate"].min(), "Última fecha": d["InvoiceDate"].max()}
    for n, d in hojas.items()
])
display(resumen_hojas)
print("Columnas:", columnas)

## Verificación e integración del bloque compartido
**Propósito:** retirar únicamente la copia entre hojas autorizada por Lourdes.

**Operaciones:** identificar facturas compartidas; comparar las ocho variables y sus multiplicidades con `value_counts(dropna=False)`; enlazar cada ocurrencia con `cumcount`; conservar la primera hoja completa e incorporar las filas restantes de la segunda. Las comprobaciones `assert` detienen la ejecución si la evidencia no coincide.

**Resultado esperado:** 22.523 copias omitidas y 1.044.848 filas integradas. La tabla `trazabilidad` vincula cada fila omitida con una conservada. La comparación incluye valores faltantes, pero no los imputa.

**Interpretación:** una factura puede tener varias líneas. No se aplica `drop_duplicates` a toda la base ni se exige que Invoice o Customer ID sean únicos por fila.

In [ ]:
compartidas = set(a["Invoice"]) & set(b["Invoice"])
bloque_a = a.loc[a["Invoice"].isin(compartidas)].copy()
bloque_b = b.loc[b["Invoice"].isin(compartidas)].copy()
conteos = pd.concat([
    bloque_a.value_counts(dropna=False).rename("primera"),
    bloque_b.value_counts(dropna=False).rename("segunda")
], axis=1).fillna(0)
assert (conteos["primera"] == conteos["segunda"]).all(), "El bloque no es una réplica exacta."
assert len(bloque_a) == len(bloque_b) == 22523

bloque_a["fila_conservada"] = bloque_a.index + 2
bloque_b["fila_omitida"] = bloque_b.index + 2
for bloque in [bloque_a, bloque_b]:
    bloque["ocurrencia"] = bloque.groupby(columnas, dropna=False).cumcount()
trazabilidad = bloque_b.merge(
    bloque_a, on=columnas + ["ocurrencia"], how="left", validate="one_to_one"
)
assert trazabilidad["fila_conservada"].notna().all()
trazabilidad["hoja_conservada"] = libro.sheet_names[0]
trazabilidad["hoja_omitida"] = libro.sheet_names[1]
trazabilidad["motivo"] = "Copia exacta entre hojas verificada en ocho variables y multiplicidad"

primera = a.assign(hoja_origen=libro.sheet_names[0], fila_excel=a.index + 2)
segunda = b.assign(hoja_origen=libro.sheet_names[1], fila_excel=b.index + 2)
integrada = pd.concat([
    primera, segunda.loc[~segunda["Invoice"].isin(compartidas)]
], ignore_index=True)
assert len(integrada) == 1044848
assert integrada.loc[integrada["Invoice"].isin(compartidas), columnas].reset_index(drop=True).equals(
    a.loc[a["Invoice"].isin(compartidas), columnas].reset_index(drop=True)
)
control_integracion = pd.DataFrame({
    "Concepto": ["Filas originales", "Copias omitidas entre hojas", "Filas integradas", "Facturas compartidas"],
    "Cantidad": [len(a) + len(b), len(trazabilidad), len(integrada), len(compartidas)]
})
display(control_integracion)

## Diagnóstico de variables y completitud
**Propósito:** medir faltantes en la base ya integrada, sin confundirlos con los conteos del archivo bruto.

**Operaciones:** `dtypes` identifica tipos leídos; `isna` cuenta ausencias; `nunique` cuenta valores distintos no nulos. Se detectan también textos vacíos o con solo espacios, sin sustituirlos.

**Interpretación:** Customer ID puede aparecer como float por los faltantes, pero conceptualmente es un identificador. No debe tratarse como predictor numérico ni rellenarse automáticamente con cero. La falta de Description requiere una decisión diferente de la falta de Customer ID.

In [ ]:
perfil = pd.DataFrame(index=columnas)
perfil.index.name = "Variable"
perfil["Tipo leído"] = integrada[columnas].dtypes.astype(str)
perfil["Nulos"] = integrada[columnas].isna().sum()
perfil["Porcentaje nulos"] = (perfil["Nulos"] / len(integrada) * 100).round(4)
perfil["Valores distintos sin nulos"] = integrada[columnas].nunique(dropna=True)
perfil["Textos vacíos o espacios"] = [
    int(integrada[c].map(lambda v: isinstance(v, str) and v.strip() == "").sum())
    for c in columnas
]
display(perfil.reset_index())
print("Clientes identificables:", integrada["Customer ID"].nunique())
print("Filas con ID:", integrada["Customer ID"].notna().sum())
print("Códigos de factura distintos (incluyen cancelaciones y ajustes):", integrada["Invoice"].nunique())

## Diagnóstico de signos, cancelaciones y repeticiones
**Propósito:** identificar situaciones que requieren criterios de negocio antes de excluir registros.

**Operaciones:** construir condiciones booleanas para prefijo C, signos de cantidades y precios y repeticiones exactas de las ocho columnas originales. Las columnas de trazabilidad no participan en la comparación de duplicados.

**Interpretación:** las condiciones se superponen; no se suman como si fueran exclusiones independientes. Una cantidad negativa no prueba un error. Los códigos especiales deben revisarse con sus descripciones antes de considerarlos productos.

In [ ]:
es_cancelacion = integrada["Invoice"].astype("string").str.upper().str.startswith("C", na=False)
q = integrada["Quantity"]
p = integrada["Price"]
condiciones = {
    "Prefijo C": es_cancelacion,
    "Cantidad negativa": q < 0,
    "Cantidad cero": q == 0,
    "Precio negativo": p < 0,
    "Precio cero": p == 0,
    "Cantidad o precio no positivo": (q <= 0) | (p <= 0),
    "Cantidad negativa sin C": (q < 0) & ~es_cancelacion,
    "Prefijo C con cantidad positiva": es_cancelacion & (q > 0),
    "Repeticiones exactas excedentes": integrada.duplicated(subset=columnas)
}
incidencias = pd.DataFrame([
    {"Condición": nombre, "Filas": int(mascara.sum()),
     "Porcentaje": round(float(mascara.mean() * 100), 4)}
    for nombre, mascara in condiciones.items()
])
display(incidencias)
print("Ejemplos de precios negativos; se conservan para revisión:")
display(integrada.loc[p < 0, ["Invoice", "StockCode", "Description", "Price", "Customer ID"]])

## Construcción del importe por línea
**Propósito:** obtener una magnitud monetaria calculable a partir de los datos originales.

**Definición:** `Importe_linea = Quantity × Price`, expresado en GBP (libras esterlinas).

**Operaciones:** multiplicación fila a fila, sin excluir signos negativos ni redondear los valores usados en cálculos. El importe calculado aún incluye ajustes y cancelaciones.

**Interpretación:** el importe por línea no es el importe de factura ni la variable objetivo de ventas futuras. Primero debemos acordar qué operaciones cuentan como ventas. Se cambia la presentación a dos decimales solo en los ejemplos visibles; no los datos almacenados.

In [ ]:
integrada["Importe_linea"] = integrada["Quantity"] * integrada["Price"]
assert integrada["Importe_linea"].notna().all()
muestra = integrada[["Invoice", "StockCode", "Quantity", "Price", "Importe_linea"]].head(8)
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(muestra)
control_importe = pd.DataFrame({
    "Signo del importe": ["Negativo", "Cero", "Positivo"],
    "Filas": [int((integrada["Importe_linea"] < 0).sum()),
              int((integrada["Importe_linea"] == 0).sum()),
              int((integrada["Importe_linea"] > 0).sum())]
})
assert control_importe["Filas"].sum() == len(integrada)
display(control_importe)

## Variables candidatas para la siguiente etapa
Estas son definiciones propuestas, no variables ya construidas ni selecciones finales del modelo.

| Variable | Definición | Nivel y condición |
|---|---|---|
| Importe por factura | Suma de importes de líneas incluidas | Factura; depende de las reglas de ventas |
| Importe histórico | Suma en la ventana anterior al corte | Cliente–corte |
| Frecuencia | Facturas de compra distintas en la ventana histórica | Cliente–corte; nunca contar filas como compras |
| Recencia | Tiempo entre el corte y la última compra anterior | Cliente–corte; definir unidad y límites |
| Importe promedio por transacción | Importe histórico / número de facturas de compra | Cliente–corte; definir tratamiento de denominador cero |
| Diversidad de productos | StockCode de productos distintos comprados | Cliente–corte; excluir conceptos no mercancía solo tras clasificación |
| Ventas futuras a 90 días | Importe incluido en el horizonte posterior completo | Etiqueta; nunca predictor |

Las ventanas, las reglas de ventas y la elegibilidad quedan pendientes. Un cliente elegible sin compras en un horizonte completamente observado puede tener etiqueta cero; un horizonte truncado por el final del archivo no permite asumir cero. Cualquier imputación o escalado aprendido deberá ajustarse solo con entrenamiento.

## Guardado de evidencia y bitácora
**Propósito:** conservar las tablas, la correspondencia de filas y la base de trabajo con el importe calculado.

**Operaciones:** exportar CSV comprimido; registrar controles y versiones; verificar que la huella del Excel no cambió; empaquetar las salidas.

**Interpretación:** el resultado es una base integrada con diagnóstico, no una base lista para entrenar. Las salidas de la sesión de Colab son temporales; descarga `Evidencias_preparacion_v01.zip` desde el panel de archivos y guarda el cuaderno en Drive.

In [ ]:
salida = Path("salidas_preparacion_v01")
salida.mkdir(exist_ok=True)
perfil.to_csv(salida / "perfil_variables.csv")
incidencias.to_csv(salida / "incidencias.csv", index=False)
control_integracion.to_csv(salida / "control_integracion.csv", index=False)
control_importe.to_csv(salida / "control_importe.csv", index=False)
trazabilidad.to_csv(salida / "trazabilidad_omitidas.csv.gz", index=False, compression="gzip")
integrada.to_csv(salida / "base_integrada_importe_v01.csv.gz", index=False, compression="gzip")
assert hashlib.sha256(ruta.read_bytes()).hexdigest() == huella_inicial
bitacora = {
    "version": "0.1", "fuente": ruta.name, "sha256_original": huella_inicial,
    "python": sys.version.split()[0], "pandas": pd.__version__, "openpyxl": openpyxl.__version__,
    "filas_integradas": len(integrada), "copias_omitidas": len(trazabilidad),
    "columnas_originales": columnas,
    "decision_confirmada": "Conservar una copia del bloque idéntico entre hojas; mantener repeticiones internas",
    "transformacion": "Importe_linea = Quantity * Price, en GBP, sin filtrar ni redondear",
    "original_intacto": True,
    "pendientes": ["Customer ID faltante", "Description faltante", "Cancelaciones y devoluciones",
                   "Ventas positivas o netas", "Conceptos especiales", "Repeticiones internas",
                   "Atípicos", "Tipos definitivos", "Ventanas y elegibilidad"]
}
(salida / "bitacora.json").write_text(json.dumps(bitacora, indent=2, ensure_ascii=False), encoding="utf-8")
paquete = Path("Evidencias_preparacion_v01.zip")
with zipfile.ZipFile(paquete, "w", zipfile.ZIP_DEFLATED) as archivo_zip:
    for archivo in sorted(salida.iterdir()):
        archivo_zip.write(archivo, arcname=archivo.name)
print("Salidas guardadas en:", salida.name)
print("Paquete para descargar:", paquete.name)
print("Original intacto. No se han entrenado modelos ni aplicado otras exclusiones.")

In [ ]:
from pathlib import Path
from google.colab import files

archivo = Path("Evidencias_preparacion_v01.zip")

if archivo.exists():
    files.download(str(archivo))
else:
    print(
        "El archivo no está en esta sesión. "
        "Vuelve a ejecutar el bloque «Guardado de evidencia y bitácora»."
    )

## Continuación: clasificación de conceptos — revisión 0.2
Los bloques anteriores corresponden al diagnóstico inicial y se conservan como antecedente. Esta continuación actualiza las decisiones: ventas netas de mercancía con descuentos D firmados; separación de clientes no identificados para el futuro modelado; exclusión de los conceptos especiales acordados del cálculo principal. No se entrenan modelos ni se define todavía una base final.

Ejecutar después de los bloques anteriores. Si la sesión se reinició, ejecutar desde la carga del Excel. La descarga de evidencias v01 corresponde solo a la etapa anterior.

## Clasificación y catálogo
**Propósito:** distinguir concepto, signo e identificación de cliente sin borrar filas.
**Operaciones:** copiar `integrada`, asignar categorías mediante códigos explícitos, marcar C por separado y construir un catálogo de todas las combinaciones código–descripción.
**Resultado esperado:** conservar 1.044.848 filas. Los códigos restantes se marcan pendientes; no se presume que todo código numérico sea mercancía.
**Interpretación:** el estado de inclusión es una regla por concepto, no una selección final: faltan validar mercancía y precios no positivos. Las repeticiones internas se conservan. No se modifican los identificadores originales.

Validación: los dos bloques analíticos se comprobaron en Python con la base integrada existente. Esta copia pública omite las salidas generadas; el bloque de exportación debe ejecutarse en la sesión de Colab.


In [ ]:
import numpy as np
clasificada = integrada.copy(deep=True)
codigo = clasificada['StockCode'].astype('string').str.strip()
reglas = {
    'POST': 'Transporte', 'DOT': 'Transporte', 'C2': 'Transporte',
    'D': 'Descuento', 'BANK CHARGES': 'Cargo/comisión',
    'AMAZONFEE': 'Cargo/comisión', 'CRUK': 'Cargo/comisión',
    'M': 'Ajuste/manual', 'm': 'Ajuste/manual', 'ADJUST': 'Ajuste/manual',
    'ADJUST2': 'Ajuste/manual', 'B': 'Ajuste/manual',
    'S': 'Muestra/prueba', 'TEST001': 'Muestra/prueba', 'TEST002': 'Muestra/prueba'
}
clasificada['tipo_concepto'] = codigo.map(reglas).fillna('Pendiente de revisión')
vale = codigo.str.lower().str.startswith('gift_', na=False) | codigo.eq('GIFT')
clasificada.loc[vale, 'tipo_concepto'] = 'Vale de regalo'
clasificada['tipo_movimiento'] = np.select(
    [clasificada['Importe_linea'] > 0, clasificada['Importe_linea'] < 0],
    ['Positivo', 'Negativo'], default='Cero')
clasificada['prefijo_C'] = clasificada['Invoice'].astype('string').str.upper().str.startswith('C', na=False)
clasificada['cliente_identificado'] = clasificada['Customer ID'].notna()
clasificada['criterio_concepto'] = 'Excluir del cálculo principal'
clasificada.loc[clasificada['tipo_concepto'].eq('Pendiente de revisión'), 'criterio_concepto'] = 'Pendiente'
clasificada.loc[clasificada['tipo_concepto'].eq('Descuento'), 'criterio_concepto'] = 'Incluir con signo y fecha registrados'
assert len(clasificada) == len(integrada) == 1044848
pd.testing.assert_frame_equal(clasificada[integrada.columns], integrada)
catalogo = clasificada.groupby(['StockCode', 'Description', 'tipo_concepto'], dropna=False).agg(
    filas=('Invoice', 'size'), documentos=('Invoice', 'nunique'),
    filas_con_cliente=('cliente_identificado', 'sum'),
    clientes_distintos=('Customer ID', 'nunique'),
    importe_GBP=('Importe_linea', 'sum')).reset_index()
control_clasificacion = clasificada.groupby(['tipo_concepto', 'criterio_concepto'], dropna=False).agg(
    filas=('Invoice', 'size'), filas_con_cliente=('cliente_identificado', 'sum'),
    importe_GBP=('Importe_linea', 'sum')).reset_index()
assert control_clasificacion.filas.sum() == len(integrada)
assert np.isclose(control_clasificacion.importe_GBP.sum(), integrada.Importe_linea.sum())
display(control_clasificacion)
print('Todas las filas e importes se conservan. No es aún la base final de ventas netas.')


## Evidencia de comisiones y descuentos
**Propósito:** reproducir las revisiones que fundamentan los criterios acordados.
**Operaciones:** filtrar CRUK y D; contar documentos, clientes y signos; buscar documentos positivos coincidentes por cliente, fecha y hora para CRUK.
**Resultado esperado:** 16 líneas CRUK por −7.933,43 GBP y 173 líneas D por −12.879,63 GBP.
**Interpretación:** una coincidencia no acredita el vínculo contable ni el beneficiario de la comisión. Los descuentos D conservan ambos signos; no cuentan como compras o productos en indicadores RFM. Los registros sin cliente permanecen en auditoría.


In [ ]:
columnas_evidencia = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Importe_linea', 'hoja_origen', 'fila_excel']
cruk = clasificada.loc[codigo.eq('CRUK'), columnas_evidencia].copy()
descuentos = clasificada.loc[codigo.eq('D'), columnas_evidencia].copy()
assert len(cruk) == 16 and np.isclose(cruk.Importe_linea.sum(), -7933.43)
assert len(descuentos) == 173 and np.isclose(descuentos.Importe_linea.sum(), -12879.63)
control_descuentos = descuentos.assign(
    con_cliente=descuentos['Customer ID'].notna(),
    signo=np.where(descuentos.Importe_linea < 0, 'Negativo', 'Positivo')
).groupby(['con_cliente', 'signo']).agg(filas=('Invoice', 'size'), importe_GBP=('Importe_linea', 'sum')).reset_index()
positivos = clasificada.loc[(clasificada.Importe_linea > 0) & ~clasificada.prefijo_C,
    ['Invoice', 'InvoiceDate', 'Customer ID']].drop_duplicates()
coincidencias_cruk = cruk.merge(positivos, on=['InvoiceDate', 'Customer ID'], how='left', suffixes=('_comision', '_positivo'))
display(cruk)
display(control_descuentos)
display(coincidencias_cruk[['Invoice_comision', 'Invoice_positivo', 'InvoiceDate', 'Customer ID', 'Importe_linea']])


## Guardado de la clasificación y sus controles
**Propósito:** conservar la base completa clasificada, catálogo, evidencias y decisiones.
**Operaciones:** exportar CSV y bitácora, empaquetar y ofrecer descarga en Colab.
**Interpretación:** la salida sustituye el estado pendiente de clasificación inicial, pero no certifica una base lista para entrenar. La revisión de mercancía, precios no positivos, ventanas y elegibilidad continúa pendiente. Los tipos originales se preservan; la estandarización definitiva de identificadores no se declara ejecutada.


In [ ]:
destino = Path('salidas_preparacion_v02')
destino.mkdir(exist_ok=True)
clasificada.to_csv(destino / 'base_clasificada_v02.csv.gz', index=False, compression='gzip')
for nombre, tabla in {'catalogo_conceptos': catalogo, 'control_clasificacion': control_clasificacion,
    'detalle_CRUK': cruk, 'detalle_descuentos': descuentos, 'control_descuentos': control_descuentos,
    'coincidencias_CRUK': coincidencias_cruk}.items():
    tabla.to_csv(destino / (nombre + '.csv'), index=False, encoding='utf-8-sig')
registro = {
    'version': '0.2', 'fecha_decisiones': '2026-10-23',
    'filas_antes': len(integrada), 'filas_despues': len(clasificada), 'filas_eliminadas_en_esta_etapa': 0,
    'acordado': ['Ventas netas de mercancía con descuentos D firmados por fecha de registro',
        'Excluir conceptos especiales acordados del cálculo principal, conservando auditoría',
        'Separar registros sin cliente del futuro modelado individual'],
    'aplicado': ['Categorías por códigos explícitos', 'Marcas de signo, C e identificación',
        'Catálogo y controles CRUK/D'],
    'pendiente': ['Clasificar códigos restantes de mercancía', 'Precios no positivos',
        'Materializar base analítica', 'Estandarización definitiva de identificadores',
        'Ventanas y elegibilidad', 'Indicadores por cliente'],
    'advertencia': 'CRUK no prueba quién recibe una comisión. C no prueba devolución física. No hay modelos entrenados.'
}
(destino / 'bitacora_v02.json').write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding='utf-8')
paquete_v02 = Path('Evidencias_preparacion_v02.zip')
with zipfile.ZipFile(paquete_v02, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(destino.iterdir()):
        z.write(f, arcname=f.name)
print('Paquete:', paquete_v02.resolve())
try:
    from google.colab import files
except ImportError:
    print('Fuera de Colab: el paquete está en la carpeta de trabajo.')
else:
    files.download(str(paquete_v02))


## Cierre de limpieza transaccional — versión 0.3
**Propósito:** construir la base monetaria por línea y mantener una auditoría separada.

**Decisiones:** conservar mercancía y descuentos D con su signo y fecha; excluir del cálculo individual filas sin cliente; separar transporte, comisiones, ajustes manuales, muestras/pruebas y vales. Se amplían las reglas con 23444 + Next Day Carriage y 22016 + Gift Voucher £100.00.

**Criterios de cierre:** separar precios no positivos sin imputarlos; no contarlos como compras. Separar las tres líneas de High Resolution Image por ambigüedad, sin afirmar que sean errores. Conservar atípicos y repeticiones internas. No excluir productos completos por una anotación de daño.

**Operaciones:** copiar integrada; asignar reglas por código y descripción; asignar un único motivo principal de separación por fila; crear base_limpia y auditoria_excluidas; conciliar filas e importes y exportar.

**Orden de motivos:** falta de cliente → concepto fuera del alcance → precio no positivo → descripción ausente → concepto ambiguo/operativo → cantidad cero. El orden evita descontar una misma fila varias veces. Las columnas originales se preservan.

**Definición operativa de mercancía:** registros con cliente, descripción y precio positivo que no corresponden a los conceptos especiales o ambiguos delimitados. Es una regla del estudio, no una validación contra un catálogo externo. No se usa el signo ni el formato numérico del código para descartar mercancía. Las anotaciones sin cliente permanecen auditables sin requerir que todas sean interpretadas individualmente.

**Resultados comprobados en Python:** 805.935 líneas conservadas, 238.913 separadas y 5.875 clientes. Importe firmado 16.397.897,407 GBP (presentar a dos decimales, conservar precisión en cálculos). Incluye 170 líneas D. No hay modelos entrenados.

**Interpretación:** base limpia a nivel transaccional, no matriz cliente-corte. Queda pendiente estandarizar identificadores, definir elegibilidad, ventanas y variables. La retención de 11.665 repeticiones excedentes en la base monetaria es una limitación documentada; no se ha ejecutado sensibilidad.

**Uso:** ejecutar después de los bloques anteriores; no usar la base clasificada como entrada de este bloque: toma integrada, conservada por el notebook. La exportación anterior v02 es un antecedente. Este bloque genera evidencias v03.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Si se ejecuta como script, cargar la base integrada original del cuaderno.
if 'integrada' not in globals():
    integrada = pd.read_csv('salidas_preparacion_v01/base_integrada_importe_v01.csv.gz', dtype={'Invoice':str,'StockCode':str})
    integrada['InvoiceDate'] = pd.to_datetime(integrada['InvoiceDate'])

limpieza = integrada.copy(deep=True)
cod = limpieza.StockCode.astype('string').str.strip()
des = limpieza.Description.astype('string').str.strip()
reglas = {'POST':'Transporte','DOT':'Transporte','C2':'Transporte',
 'BANK CHARGES':'Cargo/comisión','AMAZONFEE':'Cargo/comisión','CRUK':'Cargo/comisión',
 'M':'Ajuste/manual','m':'Ajuste/manual','ADJUST':'Ajuste/manual','ADJUST2':'Ajuste/manual','B':'Ajuste/manual',
 'S':'Muestra/prueba','TEST001':'Muestra/prueba','TEST002':'Muestra/prueba','D':'Descuento'}
limpieza['concepto_limpieza'] = cod.map(reglas).fillna('Concepto no validado')
limpieza.loc[cod.str.lower().str.startswith('gift_',na=False)|cod.eq('GIFT')|
 ((cod=='22016')&des.eq('Dotcomgiftshop Gift Voucher £100.00')), 'concepto_limpieza']='Vale de regalo'
limpieza.loc[(cod=='23444')&des.eq('Next Day Carriage'),'concepto_limpieza']='Transporte'
limpieza.loc[(cod=='23702')&des.eq('High Resolution Image'),'concepto_limpieza']='Ambiguo'
# Lista de anotaciones exactas: no buscar fragmentos como check o mould en nombres de producto.
notas = {'damaged','check','found','returned','?'}
limpieza.loc[limpieza.concepto_limpieza.eq('Concepto no validado') & des.str.lower().isin(notas), 'concepto_limpieza']='Anotación operativa/ambigua'

# Motivo principal secuencial: cada fila aparece una sola vez en la conciliación.
limpieza['motivo_limpieza']='Conservar'
def separar(mask, motivo):
    limpieza.loc[limpieza.motivo_limpieza.eq('Conservar') & mask,'motivo_limpieza']=motivo
separar(limpieza['Customer ID'].isna(),'Sin identificador de cliente')
separar(limpieza.concepto_limpieza.isin(['Transporte','Cargo/comisión','Ajuste/manual','Muestra/prueba','Vale de regalo']), 'Concepto fuera del alcance monetario')
separar(limpieza.Price <= 0, 'Precio no positivo')
separar(des.isna()|des.eq(''), 'Sin descripción interpretable')
separar(limpieza.concepto_limpieza.isin(['Ambiguo','Anotación operativa/ambigua']), 'Concepto ambiguo u operativo')
separar(limpieza.Quantity.eq(0), 'Cantidad cero')
# El residual con cliente, descripción, precio positivo y sin concepto especial se
# interpreta operativamente como mercancía; no es validación contra maestro externo.
aceptadas = limpieza.motivo_limpieza.eq('Conservar')
limpieza.loc[aceptadas & limpieza.concepto_limpieza.eq('Concepto no validado'),'concepto_limpieza']='Mercancía (definición operativa)'
base_limpia = limpieza.loc[aceptadas].copy()
auditoria_excluidas = limpieza.loc[~aceptadas].copy()
control_limpieza = limpieza.groupby('motivo_limpieza',sort=False).agg(
 filas=('Invoice','size'),importe_GBP=('Importe_linea','sum'),
 clientes_distintos=('Customer ID','nunique')).reset_index()
assert len(base_limpia)+len(auditoria_excluidas)==len(integrada)==1044848
assert np.isclose(control_limpieza.importe_GBP.sum(),integrada.Importe_linea.sum())
pd.testing.assert_frame_equal(limpieza[integrada.columns],integrada)
assert base_limpia['Customer ID'].notna().all() and base_limpia.Price.gt(0).all()
assert not base_limpia.StockCode.astype(str).eq('CRUK').any()
assert len(base_limpia[base_limpia.StockCode.astype(str).eq('D')])==170
resumen = {
 'filas_integradas':len(integrada),'filas_base_limpia':len(base_limpia),
 'filas_separadas_auditoria':len(auditoria_excluidas),
 'clientes_base_limpia':int(base_limpia['Customer ID'].nunique()),
 'importe_neto_GBP':float(base_limpia.Importe_linea.sum()),
 'lineas_negativas_conservadas':int(base_limpia.Importe_linea.lt(0).sum()),
 'repeticiones_excedentes_conservadas':int(base_limpia.duplicated(subset=['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer ID','Country']).sum()),
 'estado':'Limpieza transaccional cerrada para definición operativa; no es matriz cliente-corte',
 'reglas':'Conservar mercancía y D firmados por fecha; separar faltantes ID, conceptos excluidos, precios no positivos y ambigüedades. No eliminar atípicos ni repeticiones internas.',
 'limitaciones':['Sin maestro externo de productos: mercancía es una definición operativa residual tras controles explícitos.',
 'No todas las notas operativas de la auditoría sin cliente se clasifican semánticamente; no intervienen en el modelo individual.',
 'Ventas netas excluyen transporte, comisiones, ajustes manuales, pruebas, muestras y vales.',
 'Las repeticiones internas se conservan por falta de identificador de línea; sensibilidad pendiente en modelado.'],
 'siguiente':'Estandarización de tipos e identificadores, diccionario definitivo y variables cliente-corte; sin entrenamiento.'}
print(control_limpieza.to_string(index=False))
print(json.dumps(resumen,ensure_ascii=False,indent=2))
out=Path('salidas_limpieza_v03');out.mkdir(exist_ok=True)
base_limpia.to_csv(out/'base_limpia_v03.csv.gz',index=False,compression='gzip')
auditoria_excluidas.to_csv(out/'auditoria_excluidas_v03.csv.gz',index=False,compression='gzip')
control_limpieza.to_csv(out/'control_limpieza.csv',index=False,encoding='utf-8-sig')
(out/'bitacora_limpieza.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')

import zipfile
paquete_v03 = Path('Evidencias_limpieza_v03.zip')
with zipfile.ZipFile(paquete_v03, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        z.write(f, arcname=f.name)
print('Evidencias:', paquete_v03.resolve())
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(paquete_v03))
